In [1]:
!pip install transformers datasets torch accelerate peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 24.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="sentiment_data.csv"
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [1]:
import pandas as pd

data = {
    "text": [
        "I love this movie",
        "This product is amazing",
        "Fantastic customer service",
        "The food was delicious",
        "I am very happy with the purchase",
        "Great quality and fast delivery",
        "Excellent experience overall",
        "This app is very useful",
        "Highly recommended",
        "The book was interesting",
        "I enjoyed every moment",
        "Amazing performance",
        "The staff was friendly",
        "Best purchase ever",
        "Wonderful experience",
        "I am satisfied",
        "Very good product",
        "Works perfectly",
        "Outstanding quality",
        "Loved it",
        "I hate this service",
        "Very bad experience",
        "The product is terrible",
        "Waste of money",
        "Not worth buying",
        "Poor customer support",
        "I am disappointed",
        "Worst experience ever",
        "The app keeps crashing",
        "The food was awful",
        "Bad quality",
        "Completely useless",
        "I regret buying this",
        "Very frustrating",
        "The book was boring",
        "The staff was rude",
        "Delivery was late",
        "Not satisfied",
        "Terrible performance",
        "It stopped working",
        "Good value for money",
        "Happy with the service",
        "The design is beautiful",
        "Very reliable product",
        "The camera quality is excellent",
        "Poor battery life",
        "The screen is broken",
        "Customer service was unhelpful",
        "The software is buggy",
        "Excellent battery backup"
    ],
    "label": [
        1,1,1,1,1,1,1,1,1,1,
        1,1,1,1,1,1,1,1,1,1,
        0,0,0,0,0,0,0,0,0,0,
        0,0,0,0,0,0,0,0,0,0,
        1,1,1,1,1,
        0,0,0,0,
        1
    ]
}

df = pd.DataFrame(data)

df.head()

,text,label
0,I love this movie,1
1,This product is amazing,1
2,Fantastic customer service,1
3,The food was delicious,1
4,I am very happy with the purchase,1


In [2]:
df.to_csv("sentiment_data.csv", index=False)

In [4]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files="/content/sentiment_data.csv")

In [5]:
print(dataset["train"][0])

{'text': 'I love this movie', 'label': 1}


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21, training_loss=0.5985582442510695, metrics={'train_runtime': 996.0341, 'train_samples_per_second': 0.151, 'train_steps_per_second': 0.021, 'total_flos': 39466658304000.0, 'train_loss': 0.5985582442510695, 'epoch': 3.0})

In [9]:
model.save_pretrained("sentiment_model")
tokenizer.save_pretrained("sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('sentiment_model/tokenizer_config.json', 'sentiment_model/tokenizer.json')

In [10]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="sentiment_model",
    tokenizer="sentiment_model"
)

print(classifier("I love machine learning"))
print(classifier("This product is amazing"))
print(classifier("I hate this service"))
print(classifier("Worst experience ever"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.6527522802352905}]
[{'label': 'LABEL_1', 'score': 0.6342414617538452}]
[{'label': 'LABEL_0', 'score': 0.6546894907951355}]
[{'label': 'LABEL_0', 'score': 0.6622676253318787}]


In [13]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="sentiment_model",
    tokenizer="sentiment_model"
)

def predict(text):
    result = classifier(text)[0]

    sentiment = (
        "Positive 😊"
        if result["label"] == "LABEL_1"
        else "Negative 😞"
    )

    print("Text:", text)
    print("Sentiment:", sentiment)
    print("Confidence:", round(result["score"], 4))
    print("-" * 40)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [14]:
predict("I absolutely love this product")
predict("This is the best phone I have ever used")
predict("I hate this movie")
predict("Terrible customer support")

Text: I absolutely love this product
Sentiment: Positive 😊
Confidence: 0.6376
----------------------------------------
Text: This is the best phone I have ever used
Sentiment: Positive 😊
Confidence: 0.5749
----------------------------------------
Text: I hate this movie
Sentiment: Negative 😞
Confidence: 0.6549
----------------------------------------
Text: Terrible customer support
Sentiment: Negative 😞
Confidence: 0.7085
----------------------------------------
